# 🔄 无限循环对话触发机制

> 不是一次 invoke 内部无限聊天，而是 **多次 invoke 形成一段长期 conversation**，靠 checkpointer 保存状态。

## 🔑 核心机制

靠的是：

```python
MemorySaver()       # checkpointer
thread_id           # 同一个 thread
```

把每次 invoke 后的 state 保存下来。

---

## 📋 逐步演示

同一个 `thread_id` 连续调用：

### 第 1 次 invoke
```text
输入 Human m1 → conversation 生成 AI m2
messages = [m1, m2]  |  len = 2 → 不 summarize
```

### 第 2 次 invoke
```text
恢复 [m1, m2] → 加入 Human m3 → conversation 生成 AI m4
messages = [m1, m2, m3, m4]  |  len = 4 → 不 summarize
```

### 第 3 次 invoke
```text
恢复 [m1, m2, m3, m4] → 加入 Human m5 → conversation 生成 AI m6
messages = [m1, m2, m3, m4, m5, m6]  |  len = 6 → 不 summarize
```

### 第 4 次 invoke ⚡
```text
恢复 [m1, m2, m3, m4, m5, m6] → 加入 Human m7 → conversation 生成 AI m8
messages = [m1, m2, m3, m4, m5, m6, m7, m8]  |  len = 8 → 🔥 触发 summarize_conversation
```

---

## ✂️ summarize_conversation 做了什么？

1. 把 m1-m8 总结成 summary
2. 删除旧 messages，只保留最后 2 条

最后 state 变成：

```python
{
    "summary": "之前对话的总结...",
    "messages": [m7, m8]
}
```

---

## 🧠 下一次 invoke 时模型看到什么？

`call_model` 会看到 summary：

```python
messages = [
    SystemMessage(content="Summary of conversation earlier: ..."),
] + state["messages"]
```

所以模型看到的是：

```text
System: 之前对话总结
Human: 最近一条问题
AI: 最近一次回答
Human: 新问题
```

---

## 💡 设计意义

> 每次调用只跑一轮，但很多次调用共享同一个长期 state。
> 当历史 messages 累积太多时，**自动压缩成 summary**，避免上下文越来越长。

| 概念 | 说明 |
|------|------|
| 不是 | 一次 invoke 内部无限聊天 |
| 而是 | 多次 invoke 形成一段长期 conversation |
| 靠什么 | checkpointer 保存状态 |
| 优化 | 自动 summarize 防止上下文爆炸 💥 |